![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 1. Parte B: Modelo, entrenamiento y evaluación

**Asignatura:** Modelos de Lenguaje
**Bloque:** 1 — Introducción a las Redes Neuronales

---

## Introducción

En la Parte A convertiste texto en un tensor `(B, L)` de índices. Acá ese tensor entra por fin a un modelo.

Vas a construir un clasificador de órdenes dirigidas a un asistente virtual: recibe una frase en español —*"despertame a las nueve"*, *"qué tiempo hace mañana"*— y predice su **escenario**, que es como el corpus llama al área temática a la que apunta la orden. Son 18: `alarm`, `weather`, `music`, `iot` y catorce más. La arquitectura es la más simple que puede funcionar sobre texto: una tabla de *embeddings*, un promedio enmascarado que colapsa la secuencia en un vector, y un perceptrón multicapa encima.

La razón de empezar por acá no es que sea fácil, sino que es **completa**: tiene todas las piezas de un entrenamiento real —datos servidos por lotes, una pérdida, un optimizador, un conjunto de validación, métricas por clase, sobreajuste y regularización— y ninguna que distraiga. Cuando en la Unidad 2 le agreguemos una capa recurrente, lo único que va a cambiar es la línea que colapsa la secuencia.

Y termina con un límite. El último ejercicio te va a mostrar que este modelo no puede distinguir *"apagá la luz de la cocina"* de *"la cocina apagá de luz la"*: para él son literalmente la misma entrada. Ese techo es el que motiva toda la Unidad 2.

Al completar este laboratorio vas a poder:

- Implementar el protocolo `Dataset` de PyTorch y servir datos con `DataLoader`.
- Construir un clasificador de texto con `nn.Embedding` y promedio enmascarado.
- Verificar que un modelo sin entrenar da la pérdida que la teoría predice.
- Escribir un loop de entrenamiento completo y una función de evaluación.
- Comparar optimizadores y tasas de aprendizaje con evidencia.
- Leer una matriz de confusión de 18 clases y sacar conclusiones por clase.
- Provocar sobreajuste, medirlo, y bajarlo con *weight decay* y *dropout*.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de las Clases 2 a 6 de la Unidad 1.
- **Este laboratorio corre entero en CPU.** El entrenamiento completo son menos de treinta segundos; no hace falta GPU.
- La celda de setup deja listos el tokenizador y la clase `Vocabulario` con los que se codifica el corpus. Usalos sin modificarlos: si los reemplazás por otra implementación, los números de este laboratorio no van a coincidir con los esperados.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las dos celdas que siguen ya vienen resueltas. No hay nada que completar en ellas, pero **hay que ejecutarlas** antes de empezar y conviene leer la segunda, porque define los nombres que usan todos los ejercicios.

La primera importa las librerías. La segunda deja el corpus leído y los tensores armados.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
import os
import math
import copy
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f"Versión de PyTorch: {torch.__version__}")

### El corpus y los tensores

Esta celda lee el corpus MASSIVE en español, arma el vocabulario y deja los tensores de índices listos para entrenar.

Viene provista por dos razones. La primera es que los números de este laboratorio tienen que ser comparables entre todos: si cada uno usara su propio umbral de frecuencia o su propio largo, ninguna cifra sería contrastable con la del compañero ni con la de la solución. La segunda es que un error acá arrastraría a todo lo que sigue, y el tema de este laboratorio es otro.

Al terminar de correr vas a tener en memoria:

- `train`, `val`, `test`: los tres *splits* como `DataFrame` de pandas.
- `tok_simple` y `Vocabulario`: el tokenizador y la clase con los que se codifica el texto, disponibles por si los necesitás.
- `vocab`: el `Vocabulario` construido sobre entrenamiento con `freq_min=2`, y `L = 16`.
- `X_ent`, `X_val`, `X_test`: los tensores `(N, 16)` de índices, e `y_ent`, `y_val`, `y_test` con las etiquetas.
- `ESCENARIOS` y `N_CLASES`: los nombres de los 18 escenarios y su cantidad.

Lo último que imprime es la frecuencia de la clase mayoritaria en validación. Anotátela: es la referencia mínima contra la que se mide cualquier modelo que entrenes: acertar menos que predecir siempre la clase más común es no haber aprendido nada.

In [ ]:
# ─── Setup: el corpus y los tensores ─────────────────────────────────────────
REPO = "https://github.com/javovelez/Modelos-de-Lenguaje/raw/main/datos"
HUB  = "https://huggingface.co/datasets/SetFit/amazon_massive_scenario_es-ES/resolve/main"


def leer_split(nombre):
    """Lee un split del corpus, del repo de la materia o del Hub como respaldo."""
    try:
        return pd.read_json(f"{REPO}/{nombre}.jsonl", lines=True)
    except Exception:
        return pd.read_json(f"{HUB}/{nombre}.jsonl", lines=True)


train = leer_split("train")
val   = leer_split("validation")
test  = leer_split("test")

ESCENARIOS = (train[["label", "label_text"]]
              .drop_duplicates()
              .sort_values("label")["label_text"]
              .tolist())
N_CLASES = len(ESCENARIOS)


# El tokenizador y la clase Vocabulario vienen en un módulo auxiliar de la
# materia, que bajamos acá.
URL_PIPELINE = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/utils-v1/lab1a_pipeline.py")

if not os.path.exists("lab1a_pipeline.py"):
    urllib.request.urlretrieve(URL_PIPELINE, "lab1a_pipeline.py")

from lab1a_pipeline import tok_simple, Vocabulario


# ─── Los tensores ────────────────────────────────────────────────────────────
L = 16                                   # el 2,2% de las órdenes se trunca
vocab = Vocabulario(train.text, tokenizador=tok_simple, freq_min=2)

X_ent  = vocab.codificar_lote(train.text, L)
X_val  = vocab.codificar_lote(val.text,   L)
X_test = vocab.codificar_lote(test.text,  L)

y_ent  = torch.tensor(train.label.values)
y_val  = torch.tensor(val.label.values)
y_test = torch.tensor(test.label.values)

print(vocab)
print()
print(f"entrenamiento: X {tuple(X_ent.shape)}  y {tuple(y_ent.shape)}")
print(f"validación:    X {tuple(X_val.shape)}  y {tuple(y_val.shape)}")
print(f"prueba:        X {tuple(X_test.shape)}  y {tuple(y_test.shape)}")
print()
print(f"{N_CLASES} escenarios: {', '.join(ESCENARIOS)}")

# La clase más frecuente es la referencia mínima: un modelo que siempre
# predijera "calendar" acertaría esto. Cualquier valor por debajo es un fracaso.
mayoritaria = y_val.bincount().argmax().item()
print()
print(f"clase mayoritaria en validación: {ESCENARIOS[mayoritaria]} "
      f"({y_val.bincount().max().item() / len(y_val):.1%})")

---
## Sección A: Los datos y el modelo

Tres ejercicios para tener el modelo en pie: servir los datos por lotes, definir la arquitectura, y verificar que la arquitectura hace lo que creemos antes de entrenarla.

El tercero parece una formalidad y es de los más formativos del laboratorio: es la verificación que distingue a alguien que entiende la entropía cruzada de alguien que la invoca.

### Ejercicio 1 — El `Dataset` y los tres `DataLoader`

**Objetivo:** Implementar el protocolo `Dataset` de PyTorch y armar los tres `DataLoader`, prestando atención al orden en que el corpus viene dado.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — el `Dataset`.** Implementá la clase `OrdenesDataset(Dataset)`, que sirve pares `(tensor de índices, etiqueta)` a partir de dos tensores ya codificados:

1. `__init__(self, X, y)` guarda los dos tensores. Verificá con un `assert` que tienen el mismo largo.
2. `__len__` devuelve la cantidad de ejemplos.
3. `__getitem__(self, i)` devuelve la tupla `(X[i], y[i])`.

Instanciá los tres (`ds_ent`, `ds_val`, `ds_test`) e imprimí, para el ejemplo 0 del de entrenamiento: la forma del tensor, la etiqueta, el texto decodificado y el nombre del escenario.

> **Pista:** El protocolo `Dataset` de PyTorch es exactamente eso: una clase con `__len__` y `__getitem__`. No hay nada más. El `DataLoader` solo necesita esas dos operaciones para armar lotes, barajar e iterar.

In [ ]:
# Tu código aquí

**Parte B — los `DataLoader`.** Antes de armarlos, observá un detalle del corpus:

1. **Observá cómo viene ordenado el corpus.** Imprimí las primeras 30 etiquetas de `y_ent`. No están ordenadas por escenario —cada escenario aparece repartido a lo largo de todo el corpus—, pero tampoco vienen al azar: llegan en **rachas** de órdenes consecutivas del mismo escenario. Medilo: calculá qué proporción de los pares de órdenes consecutivas comparte escenario, y comparala contra la misma proporción sobre una permutación al azar de `y_ent`.
2. Armá `dl_ent`, `dl_val` y `dl_test` con `batch_size=64`. Decidí el valor de `shuffle` para cada uno y dejá el criterio escrito como comentario. Fijá `torch.manual_seed(0)` antes para que el barajado sea reproducible.
3. **Mostrá por qué importa:** armá además un `DataLoader` sobre `ds_ent` con `shuffle=False` e imprimí cuántas clases distintas trae su primer lote, comparado con cuántas trae el primer lote de `dl_ent`.
4. Imprimí cuántos lotes tiene cada uno de los tres `DataLoader`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Supongamos que entrenás con `shuffle=False` sobre este corpus, con las rachas que acabás de medir.

a) Describí qué le pasa al gradiente a lo largo de una época y qué forma tendría la curva de pérdida por lote.

b) ¿Por qué en validación y prueba **no** hace falta barajar? ¿Cambiaría en algo la métrica final si barajaras?

*(Escribí tu respuesta acá)*

### Ejercicio 2 — El clasificador: `nn.Embedding`, promedio enmascarado y MLP

**Objetivo:** Construir el modelo completo y ver dónde están sus parámetros.

**Enunciado:**

Todas las piezas de este modelo están en la Clase 2: la tabla de *embeddings* en su sección *4.2*, el promedio enmascarado en la *4.4* y el clasificador armado en su Parte 5. Acá lo escribís vos, sobre 18 clases en lugar de 5.

Implementá una clase `ClasificadorOrdenes`, subclase de `nn.Module`, cuyo constructor reciba en este orden: la cantidad de palabras del vocabulario (`n_vocab`), la dimensión de los *embeddings* (`dim_emb`, por defecto 64), el ancho de la capa oculta (`dim_oculta`, por defecto 128), la cantidad de clases (`n_clases`, por defecto 18) y el índice del relleno (`pad_id`, por defecto 0).

Tiene tres capas, con estos nombres:

- `self.embedding`: la tabla de *embeddings*, de `n_vocab × dim_emb`. Configurala para que la fila del relleno quede en cero y no reciba gradiente.
- `self.oculta`: una capa lineal de `dim_emb` a `dim_oculta`.
- `self.salida`: una capa lineal de `dim_oculta` a `n_clases`.

Y dos métodos:

- `promediar(self, x)`: recibe el lote de índices `(B, L)` y devuelve `(B, E)`. Es el promedio enmascarado del Ejercicio 3 de la Parte A, ahora con la tabla de *embeddings* en lugar de los vectores de ejemplo que se usaban allá.
- `forward(self, x)`: promedia, pasa el resultado por la capa oculta con una ReLU, y devuelve la salida de la última capa. **Sin softmax**: el modelo devuelve *logits*.

Después:

1. Fijá la semilla en 0, instanciá el modelo e imprimilo.
2. Imprimí una tabla con un renglón por parámetro entrenable: nombre, forma y cantidad de elementos. Cerrá con el total, y con qué porcentaje de ese total está en la tabla de *embeddings*.
3. Pasá un lote de `dl_ent` por el modelo y verificá que la forma de la salida es `(B, n_clases)`.

> **Pista 1:** Para el punto 2, `modelo.named_parameters()` recorre los parámetros dando el nombre y el tensor de cada uno.

> **Pista 2:** Que la fila del relleno quede en cero se consigue con un argumento del constructor de la capa de *embeddings*, no a mano. No es cosmético: sin él, esa fila recibiría gradiente en cada lote y se movería, con lo cual el relleno pasaría a aportar un vector no nulo al promedio.

> **Pista 3:** Que el modelo devuelva *logits* y no probabilidades es deliberado: `nn.CrossEntropyLoss` aplica el softmax internamente, de manera numéricamente estable. Aplicarlo dos veces es un error clásico.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

El 95% de los parámetros del modelo está en la tabla de *embeddings*, y solo el 5% en las dos capas lineales, que son la parte que uno llamaría "la red".

a) ¿Por qué la tabla es tan grande, y de qué dos cantidades depende su tamaño?

b) Cada fila de la tabla se actualiza solo cuando aparece su palabra en un lote. ¿Qué consecuencia tiene eso para las palabras poco frecuentes, y qué relación tiene con la decisión de `freq_min` que tomaste en la Parte A?

*(Escribí tu respuesta acá)*

### Ejercicio 3 — Cuánto tiene que dar la pérdida antes de entrenar

**Objetivo:** Predecir analíticamente cuánto tiene que dar la pérdida de un modelo recién inicializado, y comprobarlo.

**Enunciado:**

Que un modelo recién inicializado no sepa nada tiene una consecuencia que se puede calcular de antemano, y el razonamiento es corto:

- Sus pesos son valores aleatorios chicos, así que los 18 *logits* que produce para cualquier frase salen **casi iguales entre sí**.
- El softmax de 18 números casi iguales reparte la probabilidad casi en partes iguales: alrededor de $1/18$ para cada clase.
- Entonces la probabilidad que el modelo le asigna a la clase **correcta** también es $\approx 1/18$, sea cual sea esa clase y sea cual sea la frase.

Ese último valor es el único que la entropía cruzada mira. Con eso alcanza para predecir la pérdida antes de medirla.

1. **Predecí el valor.** Aplicá la fórmula de la entropía cruzada de la Clase 4 al caso en que la probabilidad de la clase correcta es $1/C$, con $C = 18$, y guardá el resultado en una variable `esperada`.
2. **Medilo.** Creá en una variable llamada `criterio` una instancia de `nn.CrossEntropyLoss`, que es la función de pérdida de la Clase 4 para clasificación multiclase: recibe los *logits* crudos y aplica el softmax internamente. Tomá después un lote de `dl_ent` y calculá con ella la pérdida del modelo sin entrenar. Como no vas a entrenar nada, hacelo sin que *autograd* registre las operaciones.

   El nombre `criterio` no es opcional: esa misma variable la van a usar los Ejercicios 4 y 5, que no la vuelven a crear.
3. **Compará** el valor medido con el predicho e imprimí la diferencia.
4. **Observá de dónde sale.** Convertí en probabilidades los 18 *logits* del primer ejemplo del lote con `F.softmax` e imprimilas, junto con su mínimo, su máximo y el $1/18$ contra el que hay que compararlas. Cuidado con el argumento `dim`: acá estás normalizando un único vector de 18 números, no un lote de vectores.

> **Pista 1:** La Clase 4 hace esta misma cuenta en su sección *1.4*, sobre las 5 clases de su corpus. La fórmula de la entropía cruzada está en la *1.2*, y el gestor de contexto que apaga *autograd*, en la *2.4*.

> **Pista 2:** El valor que predecís no depende del corpus ni de la arquitectura: sale de la cantidad de clases y de nada más. Es lo que lo vuelve una verificación tan barata.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Esta verificación cuesta dos líneas y es una de las herramientas de diagnóstico más útiles que hay. La razón es que la pérdida **se puede leer al revés**: si vale $\ell$, la probabilidad que el modelo le dio a la clase correcta fue $e^{-\ell}$.

a) Traducí así dos resultados hipotéticos de esta misma verificación: una pérdida inicial de **0,4** y una de **15**. ¿Qué probabilidad le estaría dando el modelo a la clase correcta en cada caso, y por qué ninguno de los dos números es posible en un modelo que todavía no vio ningún dato?

b) La clase mayoritaria del corpus (`calendar`) es el 13,8% de los datos, no el 5,6% que corresponde a 18 clases balanceadas. ¿Por qué la pérdida inicial da igual `ln(18)`, sin que ese desbalance la mueva?

*(Escribí tu respuesta acá)*

---
## Sección B: Entrenamiento

Tres ejercicios: el paso de descenso hecho a mano para ver el mecanismo, el loop completo, y la comparación de tasas de aprendizaje y optimizadores.

### Ejercicio 4 — Un paso de descenso de gradiente, sin optimizador

**Objetivo:** Ejecutar a mano el paso que después va a hacer el optimizador, para ver que no hay nada oculto adentro.

**Enunciado:**

La Clase 4 arma este mismo paso en su sección *3.1* y después lo repite sobre un solo lote en la *3.2*. Acá lo escribís vos, sobre este modelo y este corpus. Son cinco líneas de código y siete pasos de enunciado, porque cada línea se explica.

1. **Copiá el modelo** en una variable `m1`, para no modificar el original. Acá no sirve `clone()`: es un método de los tensores, y un `nn.Module` no lo tiene. Para copiar el modelo entero —sus tres capas y los cinco tensores de parámetros que hay adentro— se usa `copy.deepcopy`, que es de Python y no de PyTorch, y que ya viene importado en el setup.
2. **Tomá un lote** de `dl_ent`. Como necesitás uno solo y no recorrerlos todos, `next(iter(dl_ent))` te devuelve el primero y te da de una vez el par `(xb, yb)`. Calculá con `criterio` la pérdida de `m1` sobre ese lote e imprimila.
3. **Descartá los gradientes acumulados y propagá hacia atrás.** Son dos llamadas: `m1.zero_grad()` primero, y `.backward()` sobre la pérdida después. Por qué el orden importa —y por qué hay que descartar antes de calcular— está en las secciones *2.2* y *2.3* de la Clase 4.
4. **Observá un gradiente concreto.** Después de `backward()`, cada parámetro guarda el suyo en el atributo `.grad`, con la misma forma que el parámetro. Imprimí redondeado el de `m1.salida.bias`, que tiene un valor por clase.
5. **Aplicá el paso a mano**, con una tasa de aprendizaje de `0.5`: recorré los parámetros con `m1.parameters()` y restale a cada uno la tasa por su gradiente. Todo eso va adentro de un bloque `with torch.no_grad():`, que es el gestor de contexto de la sección *2.4* de la Clase 4.
6. **Recalculá la pérdida sobre el mismo lote** e imprimila. Tiene que haber bajado.
7. **Repetí el paso 30 veces sobre ese mismo lote.** Partí otra vez del modelo original, con un `deepcopy` nuevo, para que el primer valor de la serie sea el de antes de entrenar. Guardá la pérdida de cada iteración en una lista llamada `historia_lote` y graficala. Marcá con `plt.axhline` una línea horizontal punteada en el valor de la pérdida antes de entrenar que dedujiste en el Ejercicio 3.

> **Pista 1:** La razón del `no_grad()` es que actualizar los pesos no es parte del cálculo del modelo: es algo que se le hace desde afuera. Si *autograd* registrara esa operación, el grafo del paso siguiente arrastraría la historia de la actualización anterior.

> **Pista 2:** Que la pérdida sobre **un solo lote** baje hasta casi cero no es una buena noticia sobre el modelo: es memorización de esos 64 ejemplos. Acá el objetivo es solo ver el mecanismo.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

¿Qué pasaría si sacaras el bloque `torch.no_grad()` de la actualización? Explicá qué construiría `autograd` en ese caso y por qué el resultado sería incorrecto además de ineficiente.

*(Escribí tu respuesta acá)*

### Ejercicio 5 — Las funciones `evaluar()` y `entrenar()`

**Objetivo:** Escribir el loop de entrenamiento completo y la función de evaluación que vas a reusar en todo el resto del laboratorio.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — `evaluar`.** Escribí la función `evaluar(modelo, dataloader)`, que devuelve la tupla `(pérdida media, accuracy)` sobre el `dataloader` completo. Adentro de la función:

1. Poné el modelo en modo evaluación.
2. Recorré el `dataloader` sin que *autograd* registre las operaciones: no vas a propagar nada hacia atrás, y rastrearlo cuesta tiempo y memoria.
3. Acumulá la pérdida —calculada con el `criterio` que creaste en el Ejercicio 3 y **ponderada por el tamaño del lote**— y la cantidad de aciertos.
4. Devolvé los dos promedios.

Ya con la función escrita, probala sobre `dl_val` con el modelo sin entrenar: tiene que dar una pérdida cercana a la que dedujiste en el Ejercicio 3 y una accuracy cercana al azar.

> **Pista 1:** Poner el modelo en modo evaluación y devolverlo a modo entrenamiento cambia el comportamiento de capas como `Dropout`. Ahora no hay ninguna, así que no cambia nada — pero en el Ejercicio 8 sí la va a haber, y si la función no está bien escrita desde ahora, el error aparece allá y es difícil de rastrear.

> **Pista 2:** El promedio de los promedios no es el promedio. Si el último lote es más chico y contás su pérdida igual que la de los demás, esos ejemplos pesan de más; por eso hay que acumular la pérdida de cada lote multiplicada por su cantidad de ejemplos y dividir al final por el total.

In [ ]:
# Tu código aquí

**Parte B — `entrenar`.** Escribí la función `entrenar(...)`, que es el loop que arma la Clase 5 en su sección *1.2*, con dos agregados que vas a necesitar más adelante. Recibe el modelo y los `DataLoader` de entrenamiento y validación, y además cinco parámetros con estos nombres y valores por defecto: `epocas=8`, `lr=1e-3`, `optimizador="adam"`, `weight_decay=0.0` y `verbose=True`. Los vas a usar todos antes de terminar el laboratorio.

Adentro de la función:

1. Construí el optimizador según el string recibido: `"adam"` y `"sgd"` son los dos que tenés que soportar. Pasale la tasa de aprendizaje y el `weight_decay`.
2. Por cada época, poné el modelo en modo entrenamiento y recorré los lotes haciendo los tres pasos de siempre —descartar los gradientes, propagar hacia atrás, actualizar los parámetros—, guardando la pérdida de cada lote.
3. Al final de cada época evaluá sobre entrenamiento y sobre validación, y guardá las cuatro métricas.
4. Devolvé un diccionario `hist` con las claves `"ent"`, `"val"`, `"acc_ent"`, `"acc_val"` y `"por_lote"`.
5. Si `verbose`, imprimí una línea por época con las cuatro métricas.

Ya con la función escrita, y esto último fuera de ella: entrená el modelo desde cero —semilla en 0 y una instancia nueva— por 8 épocas con Adam y `lr=1e-3`. Con eso:

6. Graficá, lado a lado, la pérdida lote a lote y las dos curvas de pérdida por época. En el primer panel marcá con una línea la pérdida de referencia del Ejercicio 3.
7. Evaluá sobre `dl_test` e imprimí la pérdida y la accuracy finales, comparándolas contra la referencia de la clase mayoritaria.

> **Pista:** El `weight_decay` no hace nada hasta el Ejercicio 8, pero el optimizador lo acepta como argumento desde ahora. Dejarlo en la firma evita tener que reescribir la función más adelante.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Observá los dos gráficos y comparalos entre sí.

a) La curva por lote es mucho más ruidosa que la curva por época, aunque las dos midan la misma cantidad. Explicá por qué, y qué información da cada una que la otra no da.

b) Al final del entrenamiento la pérdida de validación es visiblemente mayor que la de entrenamiento, y la brecha se agranda con las épocas. ¿Qué está pasando, y por qué la accuracy de validación puede seguir subiendo mientras eso ocurre?

*(Escribí tu respuesta acá)*

### Ejercicio 6 — Tasa de aprendizaje y optimizador

**Objetivo:** Comparar con evidencia el efecto de la tasa de aprendizaje y del optimizador, en lugar de aceptar los valores por defecto.

**Enunciado:**

1. **Entrená cuatro configuraciones** por 5 épocas cada una. Partí siempre de un modelo nuevo y con la semilla fijada en 0, para que las cuatro partan de exactamente los mismos pesos y la comparación sea justa. Pasá `verbose=False`:

   | Configuración | optimizador | `lr` |
   |---|---|---|
   | SGD lento | `"sgd"` | 0,1 |
   | SGD razonable | `"sgd"` | 1,0 |
   | Adam razonable | `"adam"` | 1e-3 |
   | Adam demasiado alto | `"adam"` | 1e-1 |

2. **Guardá el historial de cada una** en un diccionario llamado `resultados`, con el nombre de la configuración como clave, e imprimí una tabla con la pérdida y la accuracy de validación finales de las cuatro.

3. **Graficá lado a lado** las cuatro curvas de pérdida de validación y las cuatro de accuracy de validación, por época.

> **Pista 1:** Cinco épocas de cada configuración son unos pocos segundos en CPU. No hace falta reducir el corpus.

> **Pista 2:** Atención al comparar los `lr` entre optimizadores: la tasa "razonable" de SGD y la de Adam difieren en tres órdenes de magnitud, y eso no significa que uno sea más rápido que el otro. Adam divide el paso por la magnitud del gradiente, así que su `lr` significa otra cosa.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Describí el comportamiento de cada una de las cuatro configuraciones y clasificalas en los tres comportamientos característicos de la tasa de aprendizaje: muy chica, razonable, muy grande.

b) `SGD lr=0.1` y `Adam lr=1e-3` difieren en un factor de 100 en la tasa, y sin embargo Adam avanza mucho más rápido. ¿Por qué comparar los `lr` entre optimizadores distintos no tiene sentido?

*(Escribí tu respuesta acá)*

---
## Sección C: Evaluación, sobreajuste y el límite del modelo

Ya tenés un modelo que funciona. Los últimos tres ejercicios son sobre entenderlo: qué se equivoca y por qué, cuánto está memorizando, y qué es lo que no va a poder aprender nunca con esta arquitectura.

### Ejercicio 7 — Matriz de confusión y análisis por clase

**Objetivo:** Ir más allá de la accuracy global y entender qué escenarios se confunden entre sí, y por qué.

**Enunciado:**

Volvé al `modelo` entrenado del Ejercicio 5.

1. **Escribí la función `predecir_todo(modelo, dataloader)`**, que devuelve dos tensores: las etiquetas reales y las predichas, sobre el `dataloader` completo.
2. **Escribí la función `matriz_confusion(reales, predichas, n_clases)`**, que devuelve un tensor `(n_clases, n_clases)` donde la posición `[i, j]` cuenta cuántos ejemplos de la clase real `i` se predijeron como `j`.
3. **Graficá la matriz** sobre el conjunto de prueba, como imagen, con los nombres de los escenarios en los dos ejes y una barra de color. Con 18 clases conviene una figura grande (`figsize=(9, 8)`) y las etiquetas del eje x rotadas, o no se leen.
4. **Escribí la función `precision_recall(cm)`** y mostrá una tabla con precisión, *recall*, F1 y cantidad de ejemplos por escenario, **ordenada por F1 de peor a mejor**.
5. **Listá las seis confusiones más grandes** fuera de la diagonal, en la forma `real -> predicho: cantidad`.

> **Pista 1:** Las tres funciones están en la Clase 6: `predecir_todo` y `matriz_confusion` en su Parte 2, y `precision_recall` en la Parte 3, que además explica qué pregunta contesta cada una de las dos métricas. Allá son 5 clases y acá son 18: revisá que no te quede ningún 5 escrito a mano.

> **Pista 2:** Cuidado con dividir por cero si alguna clase nunca se predice.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Observá el escenario con peor F1 y las confusiones más grandes. ¿El problema es del modelo o del esquema de clases? Justificá analizando qué frases caerían en cada uno de los escenarios involucrados.

b) Elegí una clase con **precisión alta y recall bajo** y explicá qué significa esa combinación en términos concretos. Después, observando la columna de ejemplos: ¿por qué la accuracy global del 80% esconde a las clases con F1 bajo?

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Provocar el sobreajuste, medirlo y bajarlo

**Objetivo:** Ver el sobreajuste como un fenómeno que se produce a voluntad y se mide, no como algo que "pasa", y comprobar el efecto de dos regularizadores.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

El sobreajuste y las dos herramientas para bajarlo son las Partes 4 y 5 de la Clase 6.

**Parte A — provocarlo.** Entrená un modelo deliberadamente sobredimensionado durante muchas épocas:

1. Con la semilla en 0, instanciá `ClasificadorOrdenes` con `dim_emb=128` y `dim_oculta=256`, en una variable llamada `grande`.
2. Imprimí su cantidad total de parámetros y compará contra la cantidad de ejemplos de entrenamiento.
3. Entrenalo 25 épocas con Adam y `lr=1e-3`, sin regularización, con `verbose=False`, y guardá el historial en una variable llamada `h_sin`: lo vas a volver a usar en la Parte B.
4. Imprimí una tabla con las métricas de las épocas 1, 5, 10, 15, 20 y 25.
5. Reportá la **brecha final de accuracy** (entrenamiento menos validación) y en qué época se alcanzó la mejor accuracy de validación.

In [ ]:
# Tu código aquí

**Parte B — bajarlo.** Ahora las dos herramientas:

1. Definí la clase `ClasificadorRegularizado`, con **la misma arquitectura de la Parte A** —`dim_emb=128` y `dim_oculta=256`— más una capa de *dropout* de probabilidad `p_dropout` aplicada sobre el vector promediado, antes de la capa oculta. Usá `p_dropout=0.5`.
2. Con la semilla en 0, instanciala en una variable llamada `reg`, pasándole esos dos tamaños de manera explícita, y entrenala con la misma configuración de la Parte A agregando `weight_decay=1e-4`. Guardá el historial en `h_con`.
3. Imprimí la misma tabla de épocas y la brecha final.
4. **Graficá las dos corridas juntas**: en un panel las cuatro curvas de pérdida (las dos de cada modelo), en el otro las cuatro de accuracy. Usá línea llena para validación y punteada para entrenamiento.
5. Imprimí una tabla comparativa final con, para cada modelo: accuracy de entrenamiento, accuracy de validación y brecha.

> **Pista 1:** El `dropout` va **después** del promedio y **antes** de la capa oculta. Y solo actúa en modo `.train()`: por eso importaba que `evaluar()` llame a `.eval()`.

> **Pista 2:** `weight_decay` es el nombre que PyTorch le da a la regularización L2, y se pasa directamente al constructor del optimizador.

> **Pista 3:** Al comparar, prestá atención a la accuracy de **entrenamiento** de cada modelo. La regularización casi siempre la baja: esa es exactamente su función.

> **Pista 4:** Que los dos modelos tengan exactamente el mismo tamaño no es un detalle: si el regularizado fuera más chico, la mejora que midieras sería de capacidad y no de regularización, y el ejercicio no probaría nada.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Compará las dos tablas mirando las columnas de **entrenamiento** y de **validación** por separado. ¿Qué le hizo la regularización a cada una, y por qué eso es lo que se busca?

b) En la corrida sin regularizar, la pérdida de validación toca un mínimo y después sube, mientras que la accuracy de validación se mantiene prácticamente igual. Si tuvieras que quedarte con un modelo, ¿en qué época pararías y con qué criterio?

*(Escribí tu respuesta acá)*

### Provisto: las mismas palabras en distinto orden

La celda que sigue no hay que completarla: es el experimento que da pie al último ejercicio. Le pasa al modelo tres pares de frases, y las dos de cada par están hechas con **exactamente las mismas palabras cambiadas de lugar**. Para cada par imprime qué escenario predice en cada caso, si los *logits* coinciden, y cuál es la mayor diferencia entre los dieciocho.

Usa el `modelo` que entrenaste en el Ejercicio 5, así que si todavía no lo corriste va a dar `NameError`. Ejecutala cuando llegues acá y observá bien el resultado antes de responder.

In [ ]:
# ─── Provisto: el mismo conjunto de palabras, distinto orden ────────────────
pares = [
    ("apaga la luz de la cocina",     "la cocina apaga de luz la"),
    ("recuérdame llamar a mamá",      "mamá llamar recuérdame a"),
    ("pon música y sube el volumen",  "sube el volumen y pon música"),
]

modelo.eval()
for a, b in pares:
    xa = vocab.codificar_lote([a], L)
    xb_ = vocab.codificar_lote([b], L)
    with torch.no_grad():
        la, lb = modelo(xa), modelo(xb_)

    print(f"{a!r:34s} -> {ESCENARIOS[la.argmax().item()]}")
    print(f"{b!r:34s} -> {ESCENARIOS[lb.argmax().item()]}")
    print(f"   ¿los logits son idénticos? {torch.allclose(la, lb, atol=1e-5)}")
    print(f"   máxima diferencia entre los 18 logits: {(la - lb).abs().max():.2e}")
    print()

### Ejercicio 9 — El límite de esta arquitectura

**Objetivo:** Identificar con precisión qué información descarta el modelo, y por qué eso motiva la unidad siguiente.

**Enunciado:**

La celda de arriba pasa por el modelo tres pares de frases que usan **exactamente las mismas palabras en distinto orden**. Los *logits* no se parecen: son idénticos hasta la precisión de punto flotante.

Respondé:

1. **Explicá por qué tienen que ser idénticos.** Seguí la entrada a través del `forward` y señalá la operación exacta donde se pierde el orden.

2. **Construí un ejemplo propio**, del estilo de las órdenes de este corpus, donde la invariancia cause un error grave: dos frases con el mismo conjunto de palabras y significados claramente distintos. Después decidí: ¿alcanzaría con hacer el modelo más grande —más dimensiones, más capas— para resolverlo?

3. **Proponé, conceptualmente, qué habría que cambiar** en la arquitectura para que el orden importe. No hace falta que lo implementes ni que uses el nombre técnico: describí qué propiedad tendría que tener la operación que reemplace al promedio.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] La verificación del Ejercicio 3 da un valor cercano a `ln(18) = 2,890`.
- [ ] El entrenamiento del Ejercicio 5 llega a una accuracy de validación bastante por encima del 13,8% de la clase mayoritaria.
- [ ] Todos los gráficos tienen título, etiquetas en los ejes, leyenda y grilla.
- [ ] La matriz de confusión del Ejercicio 7 tiene los nombres de los 18 escenarios en los dos ejes y se lee.
- [ ] La brecha del Ejercicio 8 baja con la regularización, y la tabla comparativa está impresa.
- [ ] Los valores numéricos que imprimo son razonables (no hay infinitos, ni `NaN`, ni valores de accuracy fuera de `[0, 1]`).
- [ ] Respondí las nueve preguntas de análisis (Ej. 1 a 9).
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Con esto cerrás el Laboratorio 1. Construiste, de punta a punta y sin cajas negras, un clasificador de texto en español:

- **El pipeline de datos**: `Dataset`, `DataLoader`, y por qué barajar el entrenamiento y no la evaluación.
- **El modelo**: `nn.Embedding`, promedio enmascarado y MLP, con el 95% de sus parámetros en la tabla de *embeddings*.
- **La verificación previa**: predecir `ln(C)` antes de entrenar, que es la manera más rápida de detectar un error de conexión entre las piezas.
- **El entrenamiento**: el paso a mano, el loop completo, y la comparación de tasas de aprendizaje y optimizadores con evidencia.
- **La evaluación**: matriz de confusión, precisión y *recall* por clase, y por qué la accuracy global esconde a las clases chicas.
- **El sobreajuste**: provocado a propósito, medido, y bajado con *weight decay* y *dropout*.

Y terminaste con un techo bien identificado: el modelo trata cada frase como una bolsa de palabras y no puede, ni en principio, distinguir *"reenviá el mensaje de mamá a papá"* de *"reenviá el mensaje de papá a mamá"*.

El **Laboratorio 2** ataca ese techo desde su base. Primero las representaciones: en lugar de dejar que la tabla de *embeddings* aprenda como subproducto de clasificar, vamos a entrenarla con un objetivo propio —word2vec, con muestreo negativo— y a comparar las dos geometrías que salen de un mismo corpus. Después, la arquitectura: una red recurrente que procesa la frase palabra por palabra y para la cual el orden, por fin, significa algo.